In [1]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate

In [2]:
import fitz
import faiss
import torch
import transformers
import sentence_transformers

print("PyMuPDF: OK")
print("FAISS: OK")
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


PyMuPDF: OK
FAISS: OK
PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving attention.pdf to attention.pdf
Saving bert.pdf to bert.pdf
Saving rag.pdf to rag.pdf


In [4]:
import os
import shutil

PDF_FOLDER = "/content/data/documents"

os.makedirs(PDF_FOLDER, exist_ok=True)

for filename in uploaded.keys():
    shutil.move(
        filename,
        os.path.join(PDF_FOLDER, filename)
    )

print("Uploaded documents:")

for filename in os.listdir(PDF_FOLDER):
    print("-", filename)

Uploaded documents:
- bert.pdf
- attention.pdf
- rag.pdf


In [5]:
import fitz
from pathlib import Path

def extract_pdf_pages(pdf_path):

    pages = []

    with fitz.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf, start=1):

            text = page.get_text("text").strip()

            if text:
                pages.append({
                    "text": text,
                    "source": Path(pdf_path).name,
                    "page": page_number
                })

    return pages

In [6]:
pdf_folder = Path(PDF_FOLDER)

all_pages = []

for pdf_path in pdf_folder.glob("*.pdf"):

    extracted = extract_pdf_pages(pdf_path)

    all_pages.extend(extracted)

    print(f"{pdf_path.name}: {len(extracted)} pages extracted")

print("\nTotal extracted pages:", len(all_pages))

bert.pdf: 16 pages extracted
attention.pdf: 15 pages extracted
rag.pdf: 19 pages extracted

Total extracted pages: 50


In [7]:
print("SOURCE:", all_pages[0]["source"])
print("PAGE:", all_pages[0]["page"])
print("\nEXTRACTED TEXT:\n")
print(all_pages[0]["text"][:2000])

SOURCE: bert.pdf
PAGE: 1

EXTRACTED TEXT:

BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin
Ming-Wei Chang
Kenton Lee
Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-
train deep bidirectional representations from
unlabeled text by jointly conditioning on both
left and right context in all layers. As a re-
sult, the pre-trained BERT model can be ﬁne-
tuned with just one additional output layer
to create state-of-the-art models for a wide
range of tasks, such as question answering and
language inference, without substantial task-
speciﬁc architecture modiﬁcations.
BERT is conceptually simple and empirically
powerful.
It obtains new state-

In [8]:
def split_into_chunks(text, chunk_size=900, overlap=150):

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size")

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks

In [9]:
all_chunks = []

for page in all_pages:

    page_chunks = split_into_chunks(
        page["text"],
        chunk_size=900,
        overlap=150
    )

    for chunk_id, chunk_text in enumerate(page_chunks):

        all_chunks.append({
            "text": chunk_text,
            "source": page["source"],
            "page": page["page"],
            "chunk_id": chunk_id,
            "chunk_uid": len(all_chunks)
        })

print("Total chunks:", len(all_chunks))

Total chunks: 245


In [10]:
for chunk in all_chunks[:3]:

    print("=" * 60)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", chunk["text"][:300])

Source: bert.pdf
Page: 1
Chunk ID: 0
Text: BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin
Ming-Wei Chang
Kenton Lee
Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands f
Source: bert.pdf
Page: 1
Chunk ID: 1
Text: ide
range of tasks, such as question answering and
language inference, without substantial task-
speciﬁc architecture modiﬁcations.
BERT is conceptually simple and empirically
powerful.
It obtains new state-of-the-art re-
sults on eleven natural language processing
tasks, including pushing the GLUE 
Source: bert.pdf
Page: 1
Chunk ID: 2
Text: and Ruder,
2018). These include sentence-level tasks such as
natural language inference (Bowman et al., 2015;
Williams et al., 2018) and paraphrasing (Dolan
and Brockett, 2005), which aim to predict the re-
lationships between sentences by analyzing them
holistically, 

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model_name = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    embedding_model_name
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (3 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
import numpy as np

texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding shape: (245, 384)


In [13]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Number of indexed vectors:", index.ntotal)

Number of indexed vectors: 245


In [14]:
import json

faiss.write_index(
    index,
    "rag_index.faiss"
)

with open("chunks.json", "w", encoding="utf-8") as f:

    json.dump(
        all_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("FAISS index saved!")
print("Chunk metadata saved!")

FAISS index saved!
Chunk metadata saved!


In [15]:
def retrieve(question, top_k=4):

    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    top_k = min(top_k, index.ntotal)

    scores, positions = index.search(
        query_vector,
        top_k
    )

    results = []

    for score, pos in zip(scores[0], positions[0]):

        if pos == -1:
            continue

        item = dict(all_chunks[pos])

        item["score"] = float(score)

        results.append(item)

    return results

In [16]:
question = "What is the Transformer architecture?"

retrieved = retrieve(question, top_k=4)

for i, item in enumerate(retrieved, start=1):

    print("=" * 70)

    print("Rank:", i)
    print("Similarity:", round(item["score"], 4))
    print("Source:", item["source"])
    print("Page:", item["page"])

    print("\nRetrieved text:")
    print(item["text"][:500])

Rank: 1
Similarity: 0.645
Source: attention.pdf
Page: 2

Retrieved text:
onjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2
Background
The goal of reducing sequential computation also forms the foundat
Rank: 2
Similarity: 0.6291
Source: attention.pdf
Page: 3

Retrieved text:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1
Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of N = 6 identi

In [17]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Qwen model loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!


In [18]:
def build_prompt(question, retrieved):

    context_parts = []

    for i, item in enumerate(retrieved, start=1):

        context_parts.append(
            f"[{i}] Source: {item['source']}, "
            f"Page: {item['page']}\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    messages = [

        {
            "role": "system",
            "content": (
                "You are an academic research assistant. "
                "Answer using only the supplied context. "
                "If the context does not contain enough "
                "information, explicitly say so. "
                "Do not invent facts. "
                "Refer to the source numbers when possible."
            )
        },

        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            )
        }

    ]

    return messages

In [20]:
def generate_answer(question, top_k=4):

    retrieved = retrieve(
        question,
        top_k=top_k
    )

    messages = build_prompt(
        question,
        retrieved
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    new_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer, retrieved

In [21]:
question = "What is the attention mechanism in Transformers?"

answer, sources = generate_answer(
    question,
    top_k=4
)

print("QUESTION:")
print(question)

print("\nGENERATED ANSWER:")
print(answer)

print("\nSOURCES:")

for item in sources:

    print(
        f"- {item['source']} "
        f"(Page {item['page']})"
    )

QUESTION:
What is the attention mechanism in Transformers?

GENERATED ANSWER:
The attention mechanism in Transformers allows each position in the encoder or decoder to attend to all positions in the previous layer of the encoder or the output of the previous decoder layer, respectively. This enables global dependencies between input and output, facilitating parallelization and improving model performance.

SOURCES:
- attention.pdf (Page 5)
- attention.pdf (Page 3)
- attention.pdf (Page 2)
- bert.pdf (Page 8)


In [29]:
!pip install -q gradio

In [30]:
def chatbot_response(message, history, top_k):

    answer, sources = generate_answer(
        message,
        top_k=int(top_k)
    )

    source_text = "\n".join([
        f"- {item['source']} | Page {item['page']} "
        f"| Similarity: {item['score']:.3f}"
        for item in sources
    ])

    response = f"""
{answer}

---

### Sources

{source_text}
"""

    return response

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # Academic RAG Assistant

        ### Ask questions about your academic research papers.

        Upload your knowledge into the RAG pipeline and receive
        context-based answers with document references.
        """
    )

    with gr.Row():

        with gr.Column(scale=4):

            chatbot = gr.Chatbot(
                height=450,
                label="Conversation"
            )

            question = gr.Textbox(
                placeholder="Ask something about your documents...",
                label="Your question"
            )

            with gr.Row():

                send_btn = gr.Button(
                    "Ask",
                    variant="primary"
                )

                clear_btn = gr.Button("Clear")

            top_k = gr.Slider(
                minimum=1,
                maximum=10,
                value=4,
                step=1,
                label="Number of retrieved passages (Top K)"
            )

    def respond(message, history, k):

        if not message.strip():
            return history, ""

        answer = chatbot_response(
            message,
            history,
            k
        )

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer}
        ]

        return history, ""

    send_btn.click(
        respond,
        inputs=[question, chatbot, top_k],
        outputs=[chatbot, question]
    )

    question.submit(
        respond,
        inputs=[question, chatbot, top_k],
        outputs=[chatbot, question]
    )

    clear_btn.click(
        lambda: ([], ""),
        outputs=[chatbot, question]
    )

demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_3761/2594261118.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://356bc524e9a252b150.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
